In [1]:
import torch
from master_model import MasterModel
from ttokenizer import Tokenizer

u_tokenizer = Tokenizer("tokenizer.json")
prompt = "the capital of united states and the capital of france"
tokens = u_tokenizer.encode(prompt)

torch.manual_seed(1)
u_model = MasterModel(vocab_size=len(u_tokenizer.vocab), embedding_dim=4, context_length=32)

sentence_meanings = u_model(tokens)
sentence_meanings.shape
#sentence_meanings_with_attention_context = u_model(tokens)
#sentence_meanings_with_attention_context

torch.Size([20, 4])

In [2]:
from transformers import Gemma3ForCausalLM

gemma_model = Gemma3ForCausalLM.from_pretrained("google/gemma-3-1b-it")


/Users/gonul/llm-from-scratch/.llm_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 340/340 [00:00<00:00, 7720.43it/s]


In [3]:
u_model, gemma_model

(MasterModel(
   (embedding): Embedding(64, 4)
   (pos_embedding): Embedding(32, 4)
 ),
 Gemma3ForCausalLM(
   (model): Gemma3TextModel(
     (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 1152, padding_idx=0)
     (layers): ModuleList(
       (0-25): 26 x Gemma3DecoderLayer(
         (self_attn): Gemma3Attention(
           (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
           (k_proj): Linear(in_features=1152, out_features=256, bias=False)
           (v_proj): Linear(in_features=1152, out_features=256, bias=False)
           (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
           (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
           (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
         )
         (mlp): Gemma3MLP(
           (gate_proj): Linear(in_features=1152, out_features=6912, bias=False)
           (up_proj): Linear(in_features=1152, out_features=6912, bias=False)
           (down_proj): Linear(in_features=6912, out_features=1152,

In [4]:
from plot_tokens import plot_dots
u_sentences = [
    {
        "words": sentence_meanings.detach().numpy(),
        "labels": u_tokenizer.tokenizer(prompt),
        "color": "red"
    }
]
plot_dots(u_sentences, "Models Context Space")

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^{\top}}{\sqrt{d_k}}\right)V
$$

$$
Q = \text{Query (Sorgu)}
$$

$$
K = \text{Key (Anahtar)}
$$

$$
V = \text{Value (İçerik)}
$$

$$
QK^{\top} \rightarrow \text{Similarity (benzerlik)}
$$

$$
\sqrt{d_k} \rightarrow \text{Scaling (sayilar cok buyumesin)}
$$

$$
\text{softmax}(\cdot) \rightarrow \text{Attention weights}
$$

$$
\text{weights} \times V \rightarrow \text{Weighted sum}
$$

$$
d_k = \text{Key vektörünün boyutu}
$$

$$
\text{embedding\_dim} = 512
$$

$$
\text{heads} = 8
$$

$$
d_k = \frac{512}{8} = 64
$$

$$
\text{Neden } \sqrt{d_k} \text{ var?}
$$

$$
\text{Dot product büyür, softmax saturate olur (çok keskin olur).}
$$

$$
\text{Scale} = \frac{1}{\sqrt{d_k}}
$$

#q icin agirliklar(sentence i agirliklar ile carp)
#k icin agirliklar(k i agirliklar ile carp cumleden olusturcaz)
#v icin agirliklar((sentence i value agirliklar ile carp))
#q (from) -> k.v(to)
#herbir kelimenin birbirine ne kadar benzedigini anliyoruz.

In [6]:
attention_weights = sentence_meanings @ sentence_meanings.T #burda ilk sentence_meanings Q, sentence_meanings.T K oldu
attention_weights.shape

torch.Size([20, 20])

In [7]:
_q_weights = torch.nn.Linear(4, 3)
_q_weights.weight, _q_weights(sentence_meanings) #bilgiyi tasiyor weightler ve bunlari egitiyoruz

(Parameter containing:
 tensor([[-0.3231,  0.3206,  0.1776,  0.1237],
         [-0.0991, -0.1560,  0.0848,  0.3869],
         [ 0.0716, -0.2364,  0.4967,  0.0107]], requires_grad=True),
 tensor([[ 3.3241e-01, -2.8807e-01,  1.7428e-01],
         [ 1.9111e-01,  2.3197e-01,  4.9175e-01],
         [-7.8087e-02, -7.0239e-01,  1.1531e-01],
         [ 1.3097e-01, -1.3319e-02,  3.7067e-01],
         [ 1.0386e-01, -2.5937e-01,  1.0479e-01],
         [ 6.8737e-01,  2.1812e-04,  1.6227e-01],
         [ 9.0142e-01,  2.6662e-02,  3.6977e-02],
         [ 2.7853e-01,  1.9267e-01,  4.7996e-01],
         [-3.5045e-02,  3.5626e-01,  8.3876e-01],
         [ 5.8966e-01,  2.7907e-01,  1.3548e+00],
         [ 3.6708e-01, -1.0401e-01,  2.8490e-01],
         [ 3.0966e-01,  1.1982e-02,  4.1809e-01],
         [ 6.6540e-01,  6.9379e-02,  2.9701e-01],
         [ 4.3007e-01,  1.4044e-01,  1.3562e+00],
         [ 8.2923e-02,  1.4064e-01,  6.1375e-01],
         [ 5.0785e-01, -2.7889e-01,  1.1391e+00],
         [ 2.8

In [8]:
q_weights = torch.nn.Linear(4, 3, bias=False)
k_weights = torch.nn.Linear(4, 3, bias=False)
v_weights = torch.nn.Linear(4, 3, bias=False)

q_of_sentence = q_weights(sentence_meanings)
k_of_sentence = k_weights(sentence_meanings)
v_of_sentence = v_weights(sentence_meanings)

q_of_sentence.shape, k_of_sentence.shape, v_of_sentence.shape 


(torch.Size([20, 3]), torch.Size([20, 3]), torch.Size([20, 3]))

In [9]:
attention_scores = q_of_sentence @ k_of_sentence.T
attention_weights = torch.softmax(attention_scores/(k_of_sentence.shape[1]**(0.5)), dim=1)

context_vector = attention_weights @ v_of_sentence
context_vector

tensor([[ 0.2033, -0.0062, -0.0066],
        [ 0.1697, -0.0510, -0.0813],
        [ 0.1754, -0.0424, -0.0664],
        [ 0.1511, -0.0476, -0.0869],
        [ 0.1587, -0.0400, -0.0735],
        [ 0.1571, -0.0192, -0.0462],
        [ 0.1308, -0.0119, -0.0511],
        [ 0.1714, -0.0470, -0.0746],
        [ 0.1416, -0.0574, -0.1042],
        [ 0.1432, -0.0563, -0.0964],
        [ 0.1483, -0.0375, -0.0744],
        [ 0.1542, -0.0438, -0.0794],
        [ 0.1649, -0.0236, -0.0472],
        [ 0.1761, -0.0680, -0.0932],
        [ 0.1667, -0.0576, -0.0910],
        [ 0.1413, -0.0523, -0.0913],
        [ 0.1477, -0.0432, -0.0818],
        [ 0.1484, -0.0419, -0.0779],
        [ 0.1602, -0.0235, -0.0490],
        [ 0.1284, -0.0499, -0.0992]], grad_fn=<MmBackward0>)

In [11]:
attention_weights.shape, context_vector.shape

(torch.Size([20, 20]), torch.Size([20, 3]))

In [12]:
attention_sentences = [
    {
        "words": q_of_sentence.detach().numpy(),
        "labels": u_tokenizer.tokenizer(prompt),
        "color": "red"
    },
    {
        "words": k_of_sentence.detach().numpy(),
        "labels": u_tokenizer.tokenizer(prompt),
        "color": "blue"
    },
     {
        "words": v_of_sentence.detach().numpy(),
        "labels": u_tokenizer.tokenizer(prompt),
        "color": "green"
    }
]
plot_dots(attention_sentences, "Query, Key, Value Space")

In [13]:
attention_sentences = [
    {
        "words": q_of_sentence.detach().numpy(),
        "labels": u_tokenizer.tokenizer(prompt),
        "color": "red"
    },
    {
        "words": k_of_sentence.detach().numpy(),
        "labels": u_tokenizer.tokenizer(prompt),
        "color": "blue"
    },
    {
        "words": v_of_sentence.detach().numpy(),
        "labels": u_tokenizer.tokenizer(prompt),
        "color": "green"
    },
    {
        "words": context_vector.detach().numpy(),
        "labels": u_tokenizer.tokenizer(prompt),
        "color": "yellow"
    }
]
plot_dots(attention_sentences, "Query, Key, Value Space and Context Vector")

In [ ]:
# new model
# SelfAttention added to MasterModel